# Global cancer trends — detailed exploratory analysis

## Ask

How do cancer incidence and mortality differ by cancer type, age, sex, country and year across the world, Poland, the United Kingdom, Spain and the United States? The notebook separates observed history, modelled 2024 estimates and demographic projections. / Jak nowotwory różnią się według typu, wieku, płci, kraju i roku?

In [1]:
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from cancer_explorer.analyse import age_peak, projection_change, rank_cancers

ROOT = Path.cwd()
if not (ROOT / 'data' / 'processed').exists():
    ROOT = ROOT.parent
DATA = ROOT / 'data' / 'processed' / 'cancer_observations.parquet'
TABLES = ROOT / 'analysis' / 'summary_tables'
FIGURES = ROOT / 'reports' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
PALETTE = {'blue': '#1f5b7a', 'gold': '#c9932f', 'ink': '#17242d', 'muted': '#87939b', 'rose': '#b14f64'}
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.titleweight': 'bold'})
data = pd.read_parquet(DATA)
len(data)

386916

## Source inventory

The canonical table preserves source version and evidence type on every record.

In [2]:
coverage = pd.read_csv(ROOT / 'data' / 'processed' / 'coverage.csv')
display(coverage)

,source_id,source_version,evidence_type,first_year,last_year,rows,geographies,cancers,sexes,age_groups,measures,metrics
0,iarc_cancer_tomorrow,GLOBOCAN 2024,projected,2024,2050,7350,5,38,3,1,2,1
1,iarc_globocan_2024,GLOBOCAN 2024,modelled,2024,2024,40800,5,37,3,20,3,5
2,who_mortality,2026-02-23,observed,1968,2024,338766,4,36,3,22,1,3


## Data quality

The strict build checks canonical-key uniqueness, uncertainty ordering, projection baselines, non-negative values and WHO total reconciliation.

In [3]:
validation = pd.read_csv(ROOT / 'data' / 'processed' / 'validation.csv')
reconciliation = pd.read_csv(ROOT / 'data' / 'processed' / 'who_reconciliation.csv')
quality_summary = {
    'canonical_records': len(data),
    'validation_issues': len(validation),
    'who_reconciliation_failures': int(reconciliation['status'].ne('pass').sum()),
    'non_missing_population_denominators': int(data['population'].notna().sum()),
}
assert quality_summary['validation_issues'] == 0
assert quality_summary['who_reconciliation_failures'] == 0
pd.Series(quality_summary, name='value')

canonical_records                      386916
validation_issues                           0
who_reconciliation_failures                 0
non_missing_population_denominators    348240
Name: value, dtype: int64

## Global burden

Current rankings use GLOBOCAN 2024 all-age counts for both sexes and exclude aggregate all-cancer categories.

In [4]:
current_numbers = data[(data.source_id == 'iarc_globocan_2024') & (data.geography_code == 'WORLD') & (data.year == 2024) & (data.sex == 'both') & (data.age_group_label == 'All ages') & (data.metric == 'number')]
top_incidence = rank_cancers(current_numbers[current_numbers.measure == 'incidence'], minimum_value=1).head(10)
top_mortality = rank_cancers(current_numbers[current_numbers.measure == 'mortality'], minimum_value=1).head(10)
display(top_incidence[['rank', 'cancer_label_en', 'cancer_label_pl', 'value']])
display(top_mortality[['rank', 'cancer_label_en', 'cancer_label_pl', 'value']])

,rank,cancer_label_en,cancer_label_pl,value
0,1,"Trachea, bronchus and lung","Tchawica, oskrzela i płuco",2637005.0
1,2,Breast,Pierś,2434087.0
2,3,Colorectum,Jelito grube i odbytnica,2041007.0
3,4,Other specified cancers,Inne określone nowotwory,1870413.0
4,5,Prostate,Gruczoł krokowy,1546112.0
5,6,Stomach,Żołądek,980286.0
6,7,Thyroid,Tarczyca,959281.0
7,8,Liver and intrahepatic bile ducts,Wątroba i wewnątrzwątrobowe drogi żółciowe,843045.0
8,9,Bladder,Pęcherz moczowy,635264.0
9,10,Cervix uteri,Szyjka macicy,604196.0


,rank,cancer_label_en,cancer_label_pl,value
0,1,"Trachea, bronchus and lung","Tchawica, oskrzela i płuco",1861839.0
1,2,Colorectum,Jelito grube i odbytnica,917895.0
2,3,Liver and intrahepatic bile ducts,Wątroba i wewnątrzwątrobowe drogi żółciowe,732489.0
3,4,Breast,Pierś,693660.0
4,5,Stomach,Żołądek,641554.0
5,6,Unspecified sites,Nowotwory o nieokreślonym umiejscowieniu,492804.0
6,7,Pancreas,Trzustka,490786.0
7,8,Other specified cancers,Inne określone nowotwory,443440.0
8,9,Oesophagus,Przełyk,441994.0
9,10,Prostate,Gruczoł krokowy,419849.0


In [5]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, table, title, color in [
    (axes[0], top_incidence.head(8), 'New cases — World, 2024', PALETTE['blue']),
    (axes[1], top_mortality.head(8), 'Deaths — World, 2024', PALETTE['rose']),
]:
    plot = table.sort_values('value')
    ax.barh(plot['cancer_label_en'], plot['value'] / 1_000_000, color=color, alpha=.9)
    ax.set_title(title, loc='left')
    ax.set_xlabel('Millions; GLOBOCAN 2024 modelled estimate')
    ax.grid(axis='x', color='#dfe5e8', linewidth=.7)
fig.tight_layout()
fig.savefig(FIGURES / 'global_top_cancers.png', bbox_inches='tight', facecolor='white')
plt.show()

## Cancer types and Countries

The same source edition and metric are used before comparing country cancer mixes.

In [6]:
country_rankings = pd.read_csv(TABLES / 'current_rankings.csv')
top_by_country = country_rankings[(country_rankings['rank'] <= 5) & (country_rankings.measure == 'incidence')]
display(top_by_country.pivot(index='rank', columns='geography_code', values='cancer_label_en'))

geography_code,ESP,GBR,POL,USA,WORLD
rank,,,,,
1,Colorectum,Breast,"Trachea, bronchus and lung",Other specified cancers,"Trachea, bronchus and lung"
2,Breast,Prostate,Colorectum,Breast,Breast
3,Prostate,Other specified cancers,Breast,Prostate,Colorectum
4,"Trachea, bronchus and lung","Trachea, bronchus and lung",Prostate,"Trachea, bronchus and lung",Other specified cancers
5,Bladder,Colorectum,Other specified cancers,Colorectum,Prostate


## Age

Age profiles use five-year age-specific rates, not raw counts.

In [7]:
age_rows = data[(data.source_id == 'iarc_globocan_2024') & (data.cancer_code == 'LUNG') & (data.measure == 'incidence') & (data.metric == 'age_specific_rate') & (data.sex == 'both')]
age_peaks = {geo: age_peak(group) for geo, group in age_rows.groupby('geography_code')}
display(pd.DataFrame(age_peaks).T)
fig, ax = plt.subplots(figsize=(10, 5.5))
for geo, group in age_rows.groupby('geography_code'):
    ax.plot(group.age_start, group.value, marker='o', markersize=3, linewidth=1.8, label=geo)
ax.set(title='Lung cancer incidence by age, 2024', xlabel='Age at start of five-year group', ylabel='Rate per 100,000')
ax.grid(color='#dfe5e8', linewidth=.7)
ax.legend(ncol=5, frameon=False, loc='upper left')
fig.tight_layout()
fig.savefig(FIGURES / 'age_profile_lung.png', bbox_inches='tight', facecolor='white')
plt.show()

,age_start,age_end,age_group_label,value
ESP,75,79,75-79,211.64
GBR,85,125,85+,412.49
POL,70,74,70-74,302.69
USA,80,84,80-84,336.42
WORLD,80,84,80-84,251.04


## Sex

Male–female comparisons use like-for-like world-standardised rates and omit sex-inapplicable cancers.

In [8]:
sex_gap = pd.read_csv(TABLES / 'sex_gap.csv')
display(sex_gap.sort_values('male_female_ratio', ascending=False).head(12)[['geography_code', 'cancer_label_en', 'measure', 'female', 'male', 'male_female_ratio']])

,geography_code,cancer_label_en,measure,female,male,male_female_ratio
167,USA,Kaposi sarcoma,incidence,0.03,0.41,13.666667
115,POL,Kaposi sarcoma,incidence,0.01,0.12,12.000000
17,ESP,Larynx,mortality,0.20,2.01,10.050000
11,ESP,Hypopharynx,mortality,0.06,0.54,9.000000
16,ESP,Larynx,incidence,0.64,5.54,8.656250
224,WORLD,Larynx,mortality,0.20,1.69,8.450000
10,ESP,Hypopharynx,incidence,0.17,1.41,8.294118
120,POL,Larynx,mortality,0.48,3.90,8.125000
114,POL,Hypopharynx,mortality,0.16,1.29,8.062500
223,WORLD,Larynx,incidence,0.44,3.44,7.818182


## Historical mortality

Observed WHO crude mortality rates extend to 1968/1969. They also reflect population ageing and may contain ICD classification breaks.

In [9]:
lung_history = data[(data.source_id == 'who_mortality') & (data.cancer_code == 'LUNG') & (data.measure == 'mortality') & (data.metric == 'crude_rate') & (data.sex == 'both') & (data.age_group_label == 'All ages')]
fig, ax = plt.subplots(figsize=(11, 5.5))
for geo, group in lung_history.groupby('geography_code'):
    ax.plot(group.year, group.value, linewidth=2, label=geo)
ax.set(title='Observed lung cancer mortality', xlabel='Year', ylabel='Crude deaths per 100,000')
ax.grid(color='#dfe5e8', linewidth=.7)
ax.legend(frameon=False, ncol=4)
fig.tight_layout()
fig.savefig(FIGURES / 'historical_lung_mortality.png', bbox_inches='tight', facecolor='white')
plt.show()

## Risks

The acquired IARC snapshot supplies cumulative incidence and mortality risk to age 74. Risk-factor attribution is not fabricated where a reproducible source export is unavailable.

In [10]:
risk = data[(data.source_id == 'iarc_globocan_2024') & (data.geography_code == 'WORLD') & (data.measure == 'lifetime_risk') & (data.sex == 'both') & (data.risk_basis == 'incidence') & ~data.cancer_code.isin(['ALL', 'ALL_EX_NMSC'])]
display(risk.nlargest(10, 'value')[['cancer_label_en', 'cancer_label_pl', 'value', 'risk_basis']])

,cancer_label_en,cancer_label_pl,value,risk_basis
346179,Breast,Pierś,5.15,incidence
346207,Prostate,Gruczoł krokowy,3.57,incidence
346163,"Trachea, bronchus and lung","Tchawica, oskrzela i płuco",2.91,incidence
346263,Colorectum,Jelito grube i odbytnica,2.13,incidence
346247,Other specified cancers,Inne określone nowotwory,1.64,incidence
346191,Cervix uteri,Szyjka macicy,1.33,incidence
346143,Stomach,Żołądek,1.03,incidence
346227,Thyroid,Tarczyca,1.02,incidence
346195,Corpus uteri,Trzon macicy,1.00,incidence
346147,Liver and intrahepatic bile ducts,Wątroba i wewnątrzwątrobowe drogi żółciowe,0.93,incidence


## Projections

Cancer Tomorrow applies the 2024 rate pattern to future populations. These are demographic scenarios, not observed forecasts.

In [11]:
projection_rows = data[(data.source_id == 'iarc_cancer_tomorrow') & (data.cancer_code == 'ALL_EX_NMSC') & (data.measure == 'incidence') & (data.sex == 'both')]
projection_summary = []
for geo, group in projection_rows.groupby('geography_code'):
    projection_summary.append({'geography_code': geo} | projection_change(group, 2050))
projection_summary = pd.DataFrame(projection_summary)
display(projection_summary[['geography_code', 'start_value', 'end_value', 'percent_change']])
plot = projection_summary[projection_summary.geography_code != 'WORLD'].sort_values('percent_change')
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.barh(plot.geography_code, plot.percent_change, color=PALETTE['gold'])
ax.set(title='Projected change in new cancer cases, 2024–2050', xlabel='Percent change; demographic scenario', ylabel='')
ax.grid(axis='x', color='#dfe5e8', linewidth=.7)
fig.tight_layout()
fig.savefig(FIGURES / 'projections_2050.png', bbox_inches='tight', facecolor='white')
plt.show()

,geography_code,start_value,end_value,percent_change
0,ESP,261994.0,343081.0,30.949945
1,GBR,415194.0,543811.0,30.977567
2,POL,196176.0,239388.0,22.027159
3,USA,1929123.0,2552769.0,32.327954
4,WORLD,19498127.0,32097025.0,64.615940


## Caveats

- Observed, modelled and projected evidence are separate.
- Separate GLOBOCAN releases must not be treated as time points in one trend.
- Crude historical rates change with population age structure.
- Registry subsets must not be relabelled as national data.
- Mortality-to-incidence ratio is not used as survival.

## Export

The compact result below is suitable for documentation and automated checks; full tables live under `analysis/summary_tables`.

In [12]:
notebook_result = {
    'records': len(data),
    'sources': sorted(data.source_id.unique()),
    'year_range': [int(data.year.min()), int(data.year.max())],
    'global_incidence_leader': top_incidence.iloc[0].cancer_label_en,
    'global_mortality_leader': top_mortality.iloc[0].cancer_label_en,
    'figures': sorted(path.name for path in FIGURES.glob('*.png')),
}
notebook_result

{'records': 386916,
 'sources': ['iarc_cancer_tomorrow', 'iarc_globocan_2024', 'who_mortality'],
 'year_range': [1968, 2050],
 'global_incidence_leader': 'Trachea, bronchus and lung',
 'global_mortality_leader': 'Trachea, bronchus and lung',
 'figures': ['age_profile_lung.png',
  'global_top_cancers.png',
  'historical_lung_mortality.png',
  'projections_2050.png']}